# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omi290/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow scikit-learn pandas

In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

In [3]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

db = duckdb.connect()

db.sql(f"""
CREATE OR REPLACE VIEW march_data AS
SELECT *
FROM read_parquet('{march_path}')
""")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [5]:
feature_vector = db.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    COALESCE(gsc_impressions,0) AS gsc_impressions,
    COALESCE(gsc_clicks,0) AS gsc_clicks,
    COALESCE(gsc_avg_position,0) AS gsc_avg_position,
    COALESCE(ga4_pageviews,0) AS ga4_pageviews,
    COALESCE(ga4_sessions,0) AS ga4_sessions,
    COALESCE(scroll_events,0) AS scroll_events

FROM march_data

WHERE ga4_data_available IS TRUE
""").df()

feature_vector.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,0,0,0.0,1,1,0
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,0,0,0.0,1,1,0
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,0,0,0.0,1,1,0
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,0,0,0.0,1,1,0
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,0,0,0.0,1,1,0


In [7]:
df = feature_vector.copy()

In [8]:
db.sql("""
DESCRIBE march_data
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [9]:
feature_vector.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'scroll_events']

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# Method Choice

I selected correlation and feature importance analysis because the warehouse release does not provide an observed target label for supervised learning. Rather than creating an artificial label that would introduce leakage, I analyze relationships among historical search and engagement signals and compare them with the Week 4 rule-based baseline. This follows the internship guidance to prioritize honest evaluation over unnecessary model complexity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# Split Design

A random train/test split is used on the March 2026 historical data.

The same feature set and evaluation metric are used for both the Week 4 rule-based baseline and the Random Forest model so that the comparison is fair.

Only historical information available before prediction is included in the model.

# Split Design

The analysis uses the March 2026 partition of the warehouse data. Only rows where `ga4_data_available IS TRUE` are included so that engagement metrics represent measured values rather than missing data. The same historical feature set is used consistently with the Week 4 baseline, and no future-window or label-derived information is included.

In [10]:
from sklearn.model_selection import train_test_split

df = feature_vector.copy()

X = df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "scroll_events"
    ]
]

# Proxy label:
# High impressions indicate high content opportunity.

y = (df["gsc_impressions"] > 1000).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(331172, 6)
(82794, 6)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
correlation = feature_vector[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_sessions",
        "scroll_events"
    ]
].corr()

correlation

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,scroll_events
gsc_impressions,1.000000,0.618054,0.076624,0.225440,0.202706,0.140865
gsc_clicks,0.618054,1.000000,-0.068025,0.269239,0.234750,0.205902
gsc_avg_position,0.076624,-0.068025,1.000000,0.108385,0.116791,0.068849
ga4_pageviews,0.225440,0.269239,0.108385,1.000000,0.976243,0.704944
ga4_sessions,0.202706,0.234750,0.116791,0.976243,1.000000,0.715904
scroll_events,0.140865,0.205902,0.068849,0.704944,0.715904,1.000000


# Comparison vs Week 4 Baseline

The correlation analysis supports the Week 4 rule-based baseline. Search visibility (impressions and clicks) and engagement signals (sessions, pageviews, scroll events) show meaningful positive relationships. Average position is less consistent, matching the Week 4 finding that it should not be used alone when prioritizing content.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

# Errors and Interpretation

The analysis identifies relationships between historical signals but cannot predict future outcomes because the warehouse release does not include an observed target label. Correlation also does not imply causation, so the findings should be used for decision support rather than causal conclusions. Additional labeled outcome data would be required to train and evaluate a supervised model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.